In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

In [ ]:
train_data = pd.read_csv("data/train.csv") 
train_data

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,e,b,y,w,t,a,f,c,b,w,...,s,w,w,p,w,o,p,k,s,m
1,p,x,y,n,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
2,e,f,s,y,t,a,f,w,n,w,...,s,w,w,p,w,o,p,n,v,d
3,p,x,f,g,f,f,f,c,b,h,...,k,p,p,p,w,o,l,h,v,g
4,e,x,f,n,f,n,f,w,b,k,...,s,w,w,p,w,o,e,k,s,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6494,p,x,y,g,f,f,f,c,b,p,...,k,p,b,p,w,o,l,h,v,d
6495,p,f,y,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,g
6496,e,x,s,w,f,n,f,w,b,n,...,f,w,w,p,w,o,e,n,a,g
6497,e,x,f,e,t,n,f,c,b,w,...,s,p,p,p,w,o,p,n,v,d


In [ ]:
test_data = pd.read_csv("data/test.csv") 
test_data

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,x,f,n,t,n,f,c,b,p,t,...,s,g,p,p,w,o,p,k,y,d
1,x,f,w,f,n,f,w,b,n,t,...,f,w,w,p,w,o,e,k,a,g
2,x,f,g,f,n,f,w,b,n,t,...,s,w,w,p,w,o,e,k,a,g
3,f,f,g,t,n,f,c,b,u,t,...,s,g,w,p,w,o,p,n,y,d
4,x,y,e,t,n,f,c,b,n,t,...,s,p,p,p,w,o,p,n,v,d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1620,k,y,n,f,f,f,c,n,b,t,...,k,w,p,p,w,o,e,w,v,d
1621,f,y,n,f,y,f,c,n,b,t,...,s,p,w,p,w,o,e,w,v,p
1622,x,y,g,f,f,f,c,b,p,e,...,k,b,n,p,w,o,l,h,v,p
1623,x,s,w,t,f,f,c,b,p,t,...,s,w,w,p,w,o,p,h,s,u


In [ ]:
y = train_data["class"]
X_train = train_data.drop(columns=["class"]).copy()

X_test = test_data.drop(columns=["class"], errors="ignore").copy()

for col in X_train.columns:
    if X_train[col].dtype == "O":
        X_train[col] = X_train[col].replace("?", np.nan).fillna("missing").astype(str)
        X_test[col]  = X_test[col].replace("?", np.nan).fillna("missing").astype(str)
    else:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col]  = X_test[col].fillna(med)

all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)
all_X_enc = pd.get_dummies(all_X)

X_train_enc = all_X_enc.iloc[:len(X_train)].copy()
X_test_enc  = all_X_enc.iloc[len(X_train):].copy()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_enc, y, test_size=0.2, random_state=42, stratify=y
)

X_tr.shape, X_val.shape

((5199, 117), (1300, 117))

In [ ]:
model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)

In [ ]:
rec = recall_score(y_val, val_pred, pos_label="p")

print("Recall (pos_label='p'):", rec)

Recall (pos_label='p'): 1.0


In [ ]:
model.fit(X_train_enc, y)

test_pred = model.predict(X_test_enc)

prediction = pd.DataFrame({"class": test_pred.astype(str)})

prediction.head()

,class
0,e
1,e
2,e
3,e
4,e
